# 05-1 · The closed-loop mediator (C2) and the veto token (C5)

| | |
|---|---|
| **What it does** | reweights the group's fairness objective from plain-language reactions, then re-picks; a veto token can remove the current pick outright |
| **Proves** | C2 — a closed loop exists, not a one-shot ranker; C5 — the veto arm is real code, not a plan |
| **Guardrail** | code performs the reweight arithmetic; nothing here asks a model to compute a number |
| **Eval surface** | the same group, run twice — with and without a veto — to see the round count change |

No API key needed for this notebook. The reaction labels below are typed
directly rather than parsed from free text — that parsing step is a genuinely
open task, tracked separately (see "What is not here yet", at the end).


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## The class, read directly from `nbio.py`

```python
class Mediator:
    def calculate_score(self, option):
        # weighted mean minus a fairness penalty on the spread
        weighted_mean = sum(w_i * s_i)
        return weighted_mean - self.lambda_fairness * variance(scores)

    def update_weights(self, reactions):
        # w_i <- w_i * exp(eta * g(reaction_i)) / Z
        g = {"accept": 0, "reject": 0.2, "strong_dislike": 0.5, "veto": 1.5}[reaction]
        ...

    def apply_veto(self, choice, reactions, candidates):
        # each user has ONE veto token; spending it removes the current pick
        ...

    def run_round(self, candidates, reactions):
        # propose -> veto (if any) -> re-propose if removed -> reweight
        ...
```
`nbio.Mediator` (this repo).


## Set up a 3-person group with a real disagreement


In [3]:
# Step 1: three diners, two who like the pick and one who does not
users = ["ann", "ben", "cara"]
satisfaction = {
    "chilli_house": {"ann": 0.95, "ben": 0.90, "cara": 0.10},
    "umami_den":    {"ann": 0.30, "ben": 0.45, "cara": 0.92},
    "sweet_stop":   {"ann": 0.20, "ben": 0.15, "cara": 0.20},
}
mediator = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
print("starting weights:", mediator.weights)
print("starting veto tokens:", mediator.veto_tokens)


starting weights: {'ann': 0.3333333333333333, 'ben': 0.3333333333333333, 'cara': 0.3333333333333333}
starting veto tokens: {'ann': 1, 'ben': 1, 'cara': 1}


## Round 1 — propose, and Cara objects


In [4]:
# Step 2: the group's first recommendation, and Cara's reaction
candidates = list(satisfaction.keys())
choice, scores = mediator.recommend(candidates)
print("proposed:", choice, "| raw scores:", {k: round(v, 3) for k, v in scores.items()})

reactions = {"ann": "accept", "ben": "accept", "cara": "strong_dislike"}
before_weights = dict(mediator.weights)
final = mediator.run_round(candidates, reactions)
nbio.io(1, "run_round (no veto)", {"weights": before_weights},
        {"weights": mediator.weights, "pick": final}, reads=["candidates", "reactions"])
print("round 1 final pick:", final)


proposed: chilli_house | raw scores: {'chilli_house': 0.574, 'umami_den': 0.522, 'sweet_stop': 0.183}
+- STEP 1  run_round (no veto)                                      [py]
| IN      : candidates, reactions
| NEW     : pick
| CHANGED : weights
+-------------------------------------------------------------------------
round 1 final pick: chilli_house


### Reading it — the objection changes the weights, and the numbers are auditable

Cara's `strong_dislike` (gradient 0.5) raised her weight relative to Ann and
Ben's `accept` (gradient 0.0). The exponentiated-gradient update is the same
arithmetic whether the reaction came from a typed label (as here) or a parsed
one — the parser is the only piece not built yet.


## Round 2 — Cara spends her veto


In [5]:
# Step 3: same candidates, but this time Cara vetoes outright
mediator2 = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
choice, _ = mediator2.recommend(candidates)
print("proposed:", choice)

reactions_veto = {"ann": "accept", "ben": "accept", "cara": "veto"}
before_tokens = dict(mediator2.veto_tokens)
final_after_veto = mediator2.run_round(list(candidates), reactions_veto)
print("Cara's veto tokens: before =", before_tokens, " after =", mediator2.veto_tokens)
print("round 1 final pick (with veto):", final_after_veto)
assert final_after_veto != choice, "the veto should have removed the original pick"
print("PASS -- vetoing the proposed option forced a different pick")


proposed: chilli_house
Cara's veto tokens: before = {'ann': 1, 'ben': 1, 'cara': 1}  after = {'ann': 1, 'ben': 1, 'cara': 0}
round 1 final pick (with veto): umami_den
PASS -- vetoing the proposed option forced a different pick


## Run several rounds — does it reach consensus?


In [6]:
# Step 4: repeat until nbio.consensus_reached() says the worst-off member is satisfied
mediator3 = nbio.Mediator(users, satisfaction, learning_rate=0.5, fairness_lambda=0.5)
candidates3 = list(satisfaction.keys())
bar = nbio.consensus_bar(satisfaction, users)   # best worst-off sat over options, minus 0.01
trace = []
for round_no in range(1, 6):
    choice, _ = mediator3.recommend(candidates3)
    worst_score = min(satisfaction[choice][u] for u in users)
    reaction = "accept" if worst_score >= bar else "strong_dislike"
    worst_user = min(users, key=lambda u: satisfaction[choice][u])
    reactions_round = {u: ("accept" if u != worst_user else reaction) for u in users}
    trace.append({"round": round_no, "pick": choice, "worst_user": worst_user,
                  "worst_score": round(worst_score, 3), "consensus": nbio.consensus_reached(worst_score, bar)})
    if nbio.consensus_reached(worst_score, bar):
        break
    mediator3.run_round(candidates3, reactions_round)

nbio.table(trace, cols=["round", "pick", "worst_user", "worst_score", "consensus"])


round  pick          worst_user  worst_score  consensus
-------------------------------------------------------
1      chilli_house  cara        0.100        False    
2      umami_den     ann         0.300        True     


### Reading it — the pass mark is relative to the group

The bar is `nbio.consensus_bar()`: the best worst-off satisfaction any option
offers (here `umami_den`, whose unhappiest member, Ann, is at 0.30) minus a 0.01
margin, so 0.29. Round 1 proposes `chilli_house`, Cara (0.10) objects, her
weight rises, and round 2 proposes `umami_den`, which clears the bar: consensus
in 2 rounds.

An earlier version used a fixed bar of 0.6. No option here clears that for
everyone, so the loop never stopped and the pick oscillated between
`chilli_house` and `umami_den`: whichever user is worst-off flips each round,
and reweighting them changes the next winner again (`eta=0.5` moves weight fast
enough to overshoot). A fixed bar is either unreachable, as here, or trivially
reached, as with real diners at 0.45, so the bar is now set per group. The
oscillation is still a real property of the reweight rule; the harness reports
how many groups hit the round cap rather than hiding them.

### What is not here yet

This notebook proves the reweight loop and the veto mechanic are real, working
code, including a real failure mode (non-convergence) worth reporting rather
than hiding. It does NOT yet include: (1) parsing a free-text reaction into one
of the four reaction labels above — currently the labels are typed directly —
and (2) wiring this loop into the actual chat graph so a real conversation
drives it. The objection parser is in **05-2** and the harness that runs the loop at
scale is in **06-evaluate**.

Next: **06-evaluate** would sweep this loop over many synthetic groups — not yet
built.


## Checks — the pass mark is relative to the group

A hand-written satisfaction matrix where everyone scores high, as real Food.com
diners do. The best worst-off option is C (0.926), so the pass mark is 0.916 and
A, the first pick, falls short. The checks assert that `nbio.consensus_bar` is
the best floor minus the margin, that with this relative bar the loop actually
negotiates (more than one round) while the old fixed bar of 0.45 stopped it in
round 1, and that `nbio.consensus_reached` cannot be called without a bar.
No data files are needed and nothing is written.

In [7]:
import inspect

USERS_CHECK = ["ann", "ben", "cara"]
S_CHECK = {
    "A": {"ann": 0.99, "ben": 0.97, "cara": 0.886},
    "B": {"ann": 0.95, "ben": 0.94, "cara": 0.905},
    "C": {"ann": 0.93, "ben": 0.93, "cara": 0.926},
}


def check_consensus_bar_is_best_floor_minus_margin():
    assert abs(nbio.consensus_bar(S_CHECK, USERS_CHECK) - (0.926 - nbio.DEFAULT_CONSENSUS_MARGIN)) < 1e-12
    assert abs(nbio.consensus_bar(S_CHECK, USERS_CHECK, 0.02) - 0.906) < 1e-12


def check_relative_bar_makes_the_loop_negotiate():
    old = nbio.mediate(S_CHECK, USERS_CHECK, bar=0.45)
    assert old["reason"] == "consensus" and old["rounds"] == 1   # a fixed low bar: no negotiation
    new = nbio.mediate(S_CHECK, USERS_CHECK)
    assert new["reason"] == "consensus" and new["rounds"] > 1
    assert new["worst_sat"] >= nbio.consensus_bar(S_CHECK, USERS_CHECK)
    assert nbio.solvable(S_CHECK, USERS_CHECK)                   # every group, by construction


def check_consensus_reached_requires_a_bar():
    assert inspect.signature(nbio.consensus_reached).parameters["bar"].default is inspect.Parameter.empty


for _check in (check_consensus_bar_is_best_floor_minus_margin,
               check_relative_bar_makes_the_loop_negotiate,
               check_consensus_reached_requires_a_bar):
    _check()
    print(f"PASS {_check.__name__}")

PASS check_consensus_bar_is_best_floor_minus_margin
PASS check_relative_bar_makes_the_loop_negotiate
PASS check_consensus_reached_requires_a_bar


## Revision: mediation quality, the veto ablation, and a utility the system does not know

Runs on the per-group matrices (`nbio.rev_data()`; Food.com diners on Uber Eats
Houston and Seattle, 200 seeded groups each) with the numpy mediation loop
`nbio.rev_mediate`, which `06-evaluate/01` checks against the harness first.

1. **E3, quality**: for one-shot Average, Fair (lambda 0.5 and 5) and
   Least-Misery, an oracle (the best mean among options that clear the bar), the
   mediator and the mediator with veto, it reports how many groups agree, in how
   many rounds, and the final pick's MinSat, mean and Jain.
2. **E4, veto ablation**: the veto split into its two parts (removal only, a larger
   gradient only), and a loop with memory (an objected pick is never re-proposed).
3. **Dominance**: how many of the mediator's final picks are Pareto-dominated in F.
4. **E5, proxy-truth gap**: the diners react with a utility that differs from the
   proxy the system sees (exact, Gaussian noise sd 0.01 / 0.02, or the order-aware
   top-3-dish utility), with graded reactions from every diner below the bar and a
   replay of the cached LLM parser's labels for real hand-labelled reactions
   (`data/objections/parser_predictions.csv`; no API calls).

Writes `figdata_rev/e3_mediation_quality.csv`, `e3_mediated_dominance.csv` and
`e5_proxy_gap.csv`; the last cell writes the LaTeX table rows
`figdata_rev/tex/tab_medquality.tex` and `tab_proxygap.tex`.

In [8]:
import csv
import numpy as np
from nbio import (REV_CITIES, REV_MARGIN, rev_data, rev_pick, rev_mediate, rev_summarise,
                  rev_parser_replay, rev_graded, rev_worst_only)

OUT = nbio.REV_OUT
D = rev_data()


def write(name, rows):
    with open(f"{OUT}/{name}", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

In [9]:
# E3: quality, and E4: the veto 2x2 ablation
rows = []
for city in REV_CITIES:
    G = D[city]
    Ss = [g["S"] for g in G]
    bars = [S.min(1).max() - REV_MARGIN for S in Ss]

    def oneshot(rule, lam=0.5, label=None):
        res = []
        for S, b in zip(Ss, bars):
            p = rev_pick(S, rule, lam)
            res.append(dict(converged=S[p].min() >= b, rounds=1, pick=p, reason="oneshot"))
        return rev_summarise(res, Ss, bars, label or rule, city)

    rows.append(oneshot("average", label="one-shot Average"))
    rows.append(oneshot("fair", 0.5, "one-shot Fair (lambda=0.5)"))
    rows.append(oneshot("fair", 5, "one-shot Fair (lambda=5)"))
    rows.append(oneshot("least_misery", label="one-shot Least-Misery"))
    # oracle: best mean among options that clear the bar (known utilities)
    res = []
    for S, b in zip(Ss, bars):
        ok = S.min(1) >= b
        p = rev_pick(S, "average", mask=ok)
        res.append(dict(converged=True, rounds=1, pick=p, reason="oracle"))
    rows.append(rev_summarise(res, Ss, bars, "oracle: best mean subject to the bar", city))
    rows.append(rev_summarise([rev_mediate(S) for S in Ss], Ss, bars, "mediator (paper)", city))
    rows.append(rev_summarise([rev_mediate(S, veto_remove=True, veto_grad=True) for S in Ss], Ss, bars,
                          "mediator + veto (paper)", city))
    # E4: veto 2x2 ablation
    rows.append(rev_summarise([rev_mediate(S, veto_remove=True) for S in Ss], Ss, bars,
                          "removal only (g=0.5)", city))
    rows.append(rev_summarise([rev_mediate(S, veto_grad=True) for S in Ss], Ss, bars,
                          "larger gradient only (g=1.5, no removal)", city))
    rows.append(rev_summarise([rev_mediate(S, memory=True) for S in Ss], Ss, bars,
                          "memory: never re-propose an objected pick", city))
for r in rows:
    print(r)
write("e3_mediation_quality.csv", rows)

{'city': 'Houston, TX', 'arm': 'one-shot Average', 'n': 200, 'agree': 95, 'pct_agree': 47.5, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.8857), 'final_mean': np.float64(0.9533), 'final_jain': np.float64(0.9962)}
{'city': 'Houston, TX', 'arm': 'one-shot Fair (lambda=0.5)', 'n': 200, 'agree': 98, 'pct_agree': 49.0, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.8904), 'final_mean': np.float64(0.9532), 'final_jain': np.float64(0.9968)}
{'city': 'Houston, TX', 'arm': 'one-shot Fair (lambda=5)', 'n': 200, 'agree': 123, 'pct_agree': 61.5, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.9126), 'final_mean': np.float64(0.9498), 'final_jain': np.float64(0.9988)}
{'city': 'Houston, TX', 'arm': 'one-shot Least-Misery', 'n': 200, 'agree': 200, 'pct_agree': 100.0, 'rounds_agree': np.float64(1.0), 'rounds_

In [10]:
# dominated mediated picks
dom_rows = []
for city in REV_CITIES:
    nd, nconv, nonuni = 0, 0, 0
    for g in D[city]:
        S = g["S"]
        r = rev_mediate(S)
        p = r["pick"]
        dominated = np.any(np.all(S >= S[p], axis=1) & np.any(S > S[p], axis=1))
        nd += dominated; nonuni += r["rounds"] > 1
    dom_rows.append(dict(city=city, mediated_picks_dominated=int(nd), groups_reweighted=int(nonuni)))
    print(dom_rows[-1])
write("e3_mediated_dominance.csv", dom_rows)

{'city': 'Houston, TX', 'mediated_picks_dominated': 0, 'groups_reweighted': 102}
{'city': 'Seattle, WA', 'mediated_picks_dominated': 0, 'groups_reweighted': 105}


In [11]:
# E5: proxy-truth gap


def truth(g, kind, rng):
    S = g["S"]
    if kind == "proxy is exact":
        return S
    if kind.startswith("noise"):
        sigma = float(kind.split("=")[1])
        return S + rng.normal(0, sigma, S.shape)
    if kind == "order-aware (top-3 dishes)":
        return g["D3"]
    raise ValueError(kind)


ARMS = [
    ("one-shot Average", None),
    ("one-shot Fair (lambda=5)", None),
    ("one-shot Least-Misery", None),
    ("mediator, worst-off objects (paper rule)", dict(react_fn=rev_worst_only)),
    ("mediator, graded multi-diner reactions", dict(react_fn=rev_graded)),
    ("  + LLM parser replay", dict(react_fn=rev_graded, parser=rev_parser_replay)),
    ("  + LLM parser replay + memory", dict(react_fn=rev_graded, parser=rev_parser_replay, memory=True)),
    ("  + LLM parser replay + veto token", dict(react_fn=rev_graded, parser=rev_parser_replay,
                                                veto_remove=True, veto_grad=True)),
]
rows5 = []
for city in REV_CITIES:
    for kind in ["proxy is exact", "noise sd=0.01", "noise sd=0.02", "order-aware (top-3 dishes)"]:
        rng0 = np.random.default_rng(11)
        Us = [truth(g, kind, rng0) for g in D[city]]
        Ss = [g["S"] for g in D[city]]
        bars = [U.min(1).max() - REV_MARGIN for U in Us]
        for label, kw in ARMS:
            if kw is None:
                rule = {"one-shot Average": ("average", 0.5), "one-shot Fair (lambda=5)": ("fair", 5),
                        "one-shot Least-Misery": ("least_misery", 0.5)}[label]
                res = []
                for S, U, b in zip(Ss, Us, bars):
                    p = rev_pick(S, rule[0], rule[1])
                    res.append(dict(converged=U[p].min() >= b, rounds=1, pick=p, reason="oneshot"))
            else:
                rng = np.random.default_rng(23)
                res = [rev_mediate(S, U=U, bar=b, rng=rng, **kw) for S, U, b in zip(Ss, Us, bars)]
            rows5.append(rev_summarise(res, Us, bars, label, city, dict(truth=kind)))
            print(rows5[-1])
write("e5_proxy_gap.csv", rows5)

{'city': 'Houston, TX', 'arm': 'one-shot Average', 'n': 200, 'agree': 95, 'pct_agree': 47.5, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.8857), 'final_mean': np.float64(0.9533), 'final_jain': np.float64(0.9962), 'truth': 'proxy is exact'}
{'city': 'Houston, TX', 'arm': 'one-shot Fair (lambda=5)', 'n': 200, 'agree': 123, 'pct_agree': 61.5, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.9126), 'final_mean': np.float64(0.9498), 'final_jain': np.float64(0.9988), 'truth': 'proxy is exact'}
{'city': 'Houston, TX', 'arm': 'one-shot Least-Misery', 'n': 200, 'agree': 200, 'pct_agree': 100.0, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.9258), 'final_mean': np.float64(0.9439), 'final_jain': np.float64(0.9992), 'truth': 'proxy is exact'}


{'city': 'Houston, TX', 'arm': 'mediator, worst-off objects (paper rule)', 'n': 200, 'agree': 180, 'pct_agree': 90.0, 'rounds_agree': np.float64(2.35), 'rounds_capped': np.float64(2.92), 'exhausted': 0, 'final_minsat': np.float64(0.9197), 'final_mean': np.float64(0.9452), 'final_jain': np.float64(0.9991), 'truth': 'proxy is exact'}


{'city': 'Houston, TX', 'arm': 'mediator, graded multi-diner reactions', 'n': 200, 'agree': 169, 'pct_agree': 84.5, 'rounds_agree': np.float64(2.4), 'rounds_capped': np.float64(3.27), 'exhausted': 0, 'final_minsat': np.float64(0.9198), 'final_mean': np.float64(0.9466), 'final_jain': np.float64(0.9991), 'truth': 'proxy is exact'}


{'city': 'Houston, TX', 'arm': '  + LLM parser replay', 'n': 200, 'agree': 164, 'pct_agree': 82.0, 'rounds_agree': np.float64(2.49), 'rounds_capped': np.float64(3.48), 'exhausted': 0, 'final_minsat': np.float64(0.9177), 'final_mean': np.float64(0.9475), 'final_jain': np.float64(0.999), 'truth': 'proxy is exact'}
{'city': 'Houston, TX', 'arm': '  + LLM parser replay + memory', 'n': 200, 'agree': 192, 'pct_agree': 96.0, 'rounds_agree': np.float64(2.6), 'rounds_capped': np.float64(2.82), 'exhausted': 0, 'final_minsat': np.float64(0.9213), 'final_mean': np.float64(0.946), 'final_jain': np.float64(0.9992), 'truth': 'proxy is exact'}
{'city': 'Houston, TX', 'arm': '  + LLM parser replay + veto token', 'n': 200, 'agree': 177, 'pct_agree': 88.5, 'rounds_agree': np.float64(2.28), 'rounds_capped': np.float64(2.94), 'exhausted': 0, 'final_minsat': np.float64(0.9206), 'final_mean': np.float64(0.9462), 'final_jain': np.float64(0.9992), 'truth': 'proxy is exact'}
{'city': 'Houston, TX', 'arm': 'one-

{'city': 'Houston, TX', 'arm': 'mediator, worst-off objects (paper rule)', 'n': 200, 'agree': 130, 'pct_agree': 65.0, 'rounds_agree': np.float64(2.91), 'rounds_capped': np.float64(4.69), 'exhausted': 0, 'final_minsat': np.float64(0.9179), 'final_mean': np.float64(0.9451), 'final_jain': np.float64(0.999), 'truth': 'noise sd=0.01'}
{'city': 'Houston, TX', 'arm': 'mediator, graded multi-diner reactions', 'n': 200, 'agree': 113, 'pct_agree': 56.5, 'rounds_agree': np.float64(2.94), 'rounds_capped': np.float64(5.14), 'exhausted': 0, 'final_minsat': np.float64(0.9179), 'final_mean': np.float64(0.947), 'final_jain': np.float64(0.999), 'truth': 'noise sd=0.01'}
{'city': 'Houston, TX', 'arm': '  + LLM parser replay', 'n': 200, 'agree': 108, 'pct_agree': 54.0, 'rounds_agree': np.float64(3.13), 'rounds_capped': np.float64(5.37), 'exhausted': 0, 'final_minsat': np.float64(0.9164), 'final_mean': np.float64(0.9467), 'final_jain': np.float64(0.9989), 'truth': 'noise sd=0.01'}


{'city': 'Houston, TX', 'arm': '  + LLM parser replay + memory', 'n': 200, 'agree': 181, 'pct_agree': 90.5, 'rounds_agree': np.float64(3.4), 'rounds_capped': np.float64(3.84), 'exhausted': 0, 'final_minsat': np.float64(0.9232), 'final_mean': np.float64(0.9478), 'final_jain': np.float64(0.9991), 'truth': 'noise sd=0.01'}
{'city': 'Houston, TX', 'arm': '  + LLM parser replay + veto token', 'n': 200, 'agree': 115, 'pct_agree': 57.5, 'rounds_agree': np.float64(2.78), 'rounds_capped': np.float64(5.0), 'exhausted': 0, 'final_minsat': np.float64(0.9194), 'final_mean': np.float64(0.9462), 'final_jain': np.float64(0.9991), 'truth': 'noise sd=0.01'}
{'city': 'Houston, TX', 'arm': 'one-shot Average', 'n': 200, 'agree': 23, 'pct_agree': 11.5, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.8795), 'final_mean': np.float64(0.9528), 'final_jain': np.float64(0.9959), 'truth': 'noise sd=0.02'}
{'city': 'Houston, TX', 'arm': 'one-shot Fair 

... (10 outputs left out)

{'city': 'Seattle, WA', 'arm': '  + LLM parser replay + memory', 'n': 200, 'agree': 132, 'pct_agree': 66.0, 'rounds_agree': np.float64(3.7), 'rounds_capped': np.float64(5.16), 'exhausted': 0, 'final_minsat': np.float64(0.9207), 'final_mean': np.float64(0.9523), 'final_jain': np.float64(0.9989), 'truth': 'noise sd=0.02'}
{'city': 'Seattle, WA', 'arm': '  + LLM parser replay + veto token', 'n': 200, 'agree': 76, 'pct_agree': 38.0, 'rounds_agree': np.float64(2.97), 'rounds_capped': np.float64(6.09), 'exhausted': 0, 'final_minsat': np.float64(0.9184), 'final_mean': np.float64(0.9503), 'final_jain': np.float64(0.9989), 'truth': 'noise sd=0.02'}
{'city': 'Seattle, WA', 'arm': 'one-shot Average', 'n': 200, 'agree': 6, 'pct_agree': 3.0, 'rounds_agree': np.float64(1.0), 'rounds_capped': np.float64(1.0), 'exhausted': 0, 'final_minsat': np.float64(0.9241), 'final_mean': np.float64(0.9612), 'final_jain': np.float64(0.9988), 'truth': 'order-aware (top-3 dishes)'}
{'city': 'Seattle, WA', 'arm': 'one

{'city': 'Seattle, WA', 'arm': 'mediator, worst-off objects (paper rule)', 'n': 200, 'agree': 33, 'pct_agree': 16.5, 'rounds_agree': np.float64(4.09), 'rounds_capped': np.float64(7.36), 'exhausted': 0, 'final_minsat': np.float64(0.936), 'final_mean': np.float64(0.9633), 'final_jain': np.float64(0.9994), 'truth': 'order-aware (top-3 dishes)'}
{'city': 'Seattle, WA', 'arm': 'mediator, graded multi-diner reactions', 'n': 200, 'agree': 22, 'pct_agree': 11.0, 'rounds_agree': np.float64(3.41), 'rounds_capped': np.float64(7.5), 'exhausted': 0, 'final_minsat': np.float64(0.9353), 'final_mean': np.float64(0.9625), 'final_jain': np.float64(0.9993), 'truth': 'order-aware (top-3 dishes)'}


{'city': 'Seattle, WA', 'arm': '  + LLM parser replay', 'n': 200, 'agree': 23, 'pct_agree': 11.5, 'rounds_agree': np.float64(3.52), 'rounds_capped': np.float64(7.48), 'exhausted': 0, 'final_minsat': np.float64(0.9342), 'final_mean': np.float64(0.9638), 'final_jain': np.float64(0.9991), 'truth': 'order-aware (top-3 dishes)'}
{'city': 'Seattle, WA', 'arm': '  + LLM parser replay + memory', 'n': 200, 'agree': 61, 'pct_agree': 30.5, 'rounds_agree': np.float64(4.25), 'rounds_capped': np.float64(6.86), 'exhausted': 0, 'final_minsat': np.float64(0.9317), 'final_mean': np.float64(0.9629), 'final_jain': np.float64(0.9989), 'truth': 'order-aware (top-3 dishes)'}


{'city': 'Seattle, WA', 'arm': '  + LLM parser replay + veto token', 'n': 200, 'agree': 30, 'pct_agree': 15.0, 'rounds_agree': np.float64(3.73), 'rounds_capped': np.float64(7.36), 'exhausted': 0, 'final_minsat': np.float64(0.9433), 'final_mean': np.float64(0.9671), 'final_jain': np.float64(0.9995), 'truth': 'order-aware (top-3 dishes)'}


In [12]:
# LaTeX table rows (tab_medquality.tex, tab_proxygap.tex) from the CSVs above
import pandas as pd
R = nbio.REV_OUT
T = os.path.join(nbio.REV_OUT, "tex")
os.makedirs(T, exist_ok=True)
H, S = "Houston, TX", "Seattle, WA"


def f(x, d=4):
    return f"{x:.{d}f}"


# mediation quality + veto ablation
q = pd.read_csv(f"{R}/e3_mediation_quality.csv")
order = ["one-shot Average", "one-shot Fair (lambda=0.5)", "one-shot Fair (lambda=5)", "one-shot Least-Misery",
         "oracle: best mean subject to the bar", "mediator (paper)", "mediator + veto (paper)",
         "removal only (g=0.5)", "larger gradient only (g=1.5, no removal)",
         "memory: never re-propose an objected pick"]
pretty = {"one-shot Average": "One-shot Average", "one-shot Fair (lambda=0.5)": r"One-shot Fair ($\lambda{=}0.5$)",
          "one-shot Fair (lambda=5)": r"One-shot Fair ($\lambda{=}5$)", "one-shot Least-Misery": "One-shot Least-Misery",
          "oracle: best mean subject to the bar": "Oracle: best mean that clears the bar",
          "mediator (paper)": r"Mediator ($g{=}0.5$, no removal)",
          "mediator + veto (paper)": r"Mediator + veto ($g{=}1.5$, removal)",
          "removal only (g=0.5)": r"Removal only ($g{=}0.5$)",
          "larger gradient only (g=1.5, no removal)": r"Larger gradient only ($g{=}1.5$)",
          "memory: never re-propose an objected pick": "Memory: objected picks never re-proposed"}
lines = []
for i, a in enumerate(order):
    if i in (5, 7):
        lines.append(r"\midrule")
    cells = [pretty[a]]
    for c in (H, S):
        r = q[(q.city == c) & (q.arm == a)].iloc[0]
        ra = r.rounds_agree
        cells += [f"{r.agree} ({r.pct_agree:.1f}\\%)", f"{float(ra):.2f}", f(r.final_minsat), f(r.final_mean)]
    lines.append(" & ".join(cells) + r" \\")
open(f"{T}/tab_medquality.tex", "w").write("\n".join(lines) + "\n")

# proxy gap
p = pd.read_csv(f"{R}/e5_proxy_gap.csv")
truths = ["proxy is exact", "noise sd=0.01", "noise sd=0.02", "order-aware (top-3 dishes)"]
arms = list(dict.fromkeys(p.arm))
apretty = {"one-shot Average": "One-shot Average", "one-shot Fair (lambda=5)": r"One-shot Fair ($\lambda{=}5$)",
           "one-shot Least-Misery": "One-shot Least-Misery",
           "mediator, worst-off objects (paper rule)": "Loop, worst-off diner objects (original rule)",
           "mediator, graded multi-diner reactions": "Loop, graded reactions from every diner",
           "  + LLM parser replay": r"\quad + LLM-parser replay",
           "  + LLM parser replay + memory": r"\quad + LLM-parser replay + memory",
           "  + LLM parser replay + veto token": r"\quad + LLM-parser replay + veto token"}
lines = []
for i, a in enumerate(arms):
    if i == 3:
        lines.append(r"\midrule")
    cells = [apretty[a]]
    for t in truths:
        hh = p[(p.city == H) & (p.arm == a) & (p.truth == t)].iloc[0]
        ss = p[(p.city == S) & (p.arm == a) & (p.truth == t)].iloc[0]
        cells.append(f"{hh.pct_agree:.1f} / {ss.pct_agree:.1f}")
    lines.append(" & ".join(cells) + r" \\")
open(f"{T}/tab_proxygap.tex", "w").write("\n".join(lines) + "\n")
for fn in ("tab_medquality.tex", "tab_proxygap.tex"):
    print("==", fn); print(open(f"{T}/{fn}").read())

== tab_medquality.tex
One-shot Average & 95 (47.5\%) & 1.00 & 0.8857 & 0.9533 & 94 (47.0\%) & 1.00 & 0.8910 & 0.9533 \\
One-shot Fair ($\lambda{=}0.5$) & 98 (49.0\%) & 1.00 & 0.8904 & 0.9532 & 95 (47.5\%) & 1.00 & 0.8934 & 0.9532 \\
One-shot Fair ($\lambda{=}5$) & 123 (61.5\%) & 1.00 & 0.9126 & 0.9498 & 120 (60.0\%) & 1.00 & 0.9132 & 0.9505 \\
One-shot Least-Misery & 200 (100.0\%) & 1.00 & 0.9258 & 0.9439 & 200 (100.0\%) & 1.00 & 0.9268 & 0.9439 \\
Oracle: best mean that clears the bar & 200 (100.0\%) & 1.00 & 0.9226 & 0.9461 & 200 (100.0\%) & 1.00 & 0.9236 & 0.9464 \\
\midrule
Mediator ($g{=}0.5$, no removal) & 180 (90.0\%) & 2.35 & 0.9197 & 0.9452 & 180 (90.0\%) & 2.35 & 0.9209 & 0.9457 \\
Mediator + veto ($g{=}1.5$, removal) & 187 (93.5\%) & 1.91 & 0.9204 & 0.9443 & 190 (95.0\%) & 1.99 & 0.9223 & 0.9456 \\
\midrule
Removal only ($g{=}0.5$) & 187 (93.5\%) & 2.29 & 0.9204 & 0.9452 & 185 (92.5\%) & 2.27 & 0.9211 & 0.9453 \\
Larger gradient only ($g{=}1.5$) & 183 (91.5\%) & 1.91 & 0.920

## Revision: how often does the loop without memory re-propose an objected pick?

The same loop as E5 above (graded reactions, LLM-parser replay, no memory), run
by hand so every proposal can be counted. For exact reactions and for noise with
sd 0.01, it counts the proposals that repeat a pick some diner already objected
to. Writes `figdata_rev/e5b_repropose.csv`.

In [13]:
from nbio import REV_ETA, REV_CAP, REV_GRAD

rows = []
for city in REV_CITIES:
    for kind, sd in [("exact", 0.0), ("noise sd=0.01", 0.01)]:
        rng0 = np.random.default_rng(11)
        Ss = [g["S"] for g in D[city]]
        Us = [S + (rng0.normal(0, sd, S.shape) if sd else 0) for S in Ss]
        rng = np.random.default_rng(23)
        rep, props = 0, 0
        for S, U in zip(Ss, Us):
            bar = U.min(1).max() - REV_MARGIN
            n = S.shape[1]; w = np.full(n, 1 / n); rejected = set()
            for rnd in range(REV_CAP):
                p = rev_pick(S, "fair", 0.5, w)
                props += 1
                if p in rejected: rep += 1
                u = U[p]
                if u.min() >= bar: break
                labels = [rev_parser_replay(l, rng) for l in rev_graded(u, bar, int(np.argmin(u)), rng)]
                if any(l != "accept" for l in labels): rejected.add(p)
                w = w * np.exp(REV_ETA * np.array([REV_GRAD[l] for l in labels])); w /= w.sum()
        rows.append(dict(city=city, truth=kind, proposals=props, repeated_rejected=rep,
                         pct=round(100 * rep / props, 1)))
        print(rows[-1])
with open(nbio.REV_OUT + "/e5b_repropose.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

{'city': 'Houston, TX', 'truth': 'exact', 'proposals': 697, 'repeated_rejected': 334, 'pct': 47.9}


{'city': 'Houston, TX', 'truth': 'noise sd=0.01', 'proposals': 1074, 'repeated_rejected': 623, 'pct': 58.0}


{'city': 'Seattle, WA', 'truth': 'exact', 'proposals': 703, 'repeated_rejected': 335, 'pct': 47.7}


{'city': 'Seattle, WA', 'truth': 'noise sd=0.01', 'proposals': 1066, 'repeated_rejected': 651, 'pct': 61.1}


## Revision: a live AI diner instead of the parser replay (E8)

E5 above feeds the loop a *replay*: each intended label is swapped for a cached
gpt-oss-20b reading of a hand-written sentence with that gold label. Here the
conversation is live. Every round, every diner's intended label (the same graded
rule, `nbio.rev_graded`) is written as a chat message by **gpt-4o-mini**
(`nbio.write_diner_message`, prompt `DINER_PROMPT_VERSION` v2), and
**`nbio.parse_objection`** (gpt-oss-20b on Groq, prompt v1, unchanged) reads it
back. The loop reweights on what was *read*. Both models run at temperature 0
with seed 7.

The writer is told how the diner feels, in plain words, never a score, an outcome
or the label the reader has to find. Diners are named by seat (Ann, Ben, Cara, ...)
and the suggestion is the real restaurant name. A veto still needs the diner's
token: a strong objection from a diner who still has one is said as a veto, as in
the replay's veto row.

**Why v2.** The v1 pilot (still in the log) found two problems with the writer, not
the reader. No veto message ever named the veto ("I'm not interested in going to
Dream Wings. Let's find another option."), so the parser read every veto as a
mild `reject` and the token never fired. And under "sarcastic", happy diners wrote
complaints ("Oh great, nothing says fine dining like tapioca and tea"). v2 asks a
vetoing diner to say they are using their veto, and has happy diners write plainly
in every style, so sarcasm is tested on objections only. Nothing else changed: the
strong-dislike feeling is the same, so the writer's habit of softening it ("I'm not
really feeling Moge Tee") is still part of what is measured.

The groups, noise draws (`truth()`, seed 11), bars and loop settings are E5's, so
each row of `figdata_rev/e8_live_llm.csv` lines up with the matching
`+ LLM parser replay` / `+ LLM parser replay + veto token` row of
`e5_proxy_gap.csv` (Table 8), once per writing style (plain, polite, vague,
sarcastic). The extra columns say how often the reader got the diner wrong.

Every message and its reading is appended to `runs/mediator_llm.jsonl`, which is
also the cache: with `MEDIATOR_LIVE = False` (the default) these cells make **no
API calls** and rebuild every number from the log. A missing message is reported,
not generated. Setting it to `True` needs `OPENAI_API_KEY` (capped by
`FM_OPENAI_BUDGET_USD`) and `GROQ_API_KEY`, and resumes where the log stops.

In [14]:
# Parameters
MEDIATOR_LIVE = False      # True makes real API calls and appends them to runs/mediator_llm.jsonl
RUN_FULL = False           # with MEDIATOR_LIVE: also generate the full grid below (not just the pilot)
PILOT_N = 5                # groups in the pilot
PILOT_STYLES = list(nbio.DINER_STYLES)          # plain, polite, vague, sarcastic
LIVE_STYLES = list(nbio.DINER_STYLES)
LIVE_TRUTHS = ["proxy is exact", "noise sd=0.01", "noise sd=0.02", "order-aware (top-3 dishes)"]
LIVE_WORKERS = 4           # groups run in parallel threads

In [15]:
from concurrent.futures import ThreadPoolExecutor
from nbio import DinerChat, rev_talk_stats, print_chat, catalog_names

if MEDIATOR_LIVE:
    from openai import OpenAI
    writer = nbio.retrying_client(nbio.openai_client())     # capped by FM_OPENAI_BUDGET_USD
    reader = nbio.retrying_client(OpenAI(api_key=os.environ["GROQ_API_KEY"],
                                         base_url="https://api.groq.com/openai/v1", timeout=60))
else:
    writer = reader = None
CHAT = DinerChat(writer=writer, reader=reader, live=MEDIATOR_LIVE)
print(f"live: {MEDIATOR_LIVE} | writer {CHAT.writer_model}, reader {CHAT.reader_model}, seed {CHAT.seed}")
print(f"{len(CHAT.events)} messages already in {os.path.relpath(CHAT.log_path, nbio.HERE)}")
print(f"\nwriter prompt {nbio.DINER_PROMPT_VERSION}, as Ben sees it when he really does not want to go:\n")
print(nbio.diner_prompt("strong_dislike", "Ben", ["Ann", "Cara"], "Taqueria Arandas", "plain"))
print("\nfeelings:", json.dumps(nbio.DINER_FEELINGS, indent=1))
print("styles:", json.dumps(nbio.DINER_STYLES, indent=1))

live: False | writer gpt-4o-mini, reader openai/gpt-oss-20b, seed 7
2078 messages already in runs/mediator_llm.jsonl

writer prompt v2, as Ben sees it when he really does not want to go:

You are Ben, out with Ann and Cara to choose a restaurant together.
A group mediator has just suggested: Taqueria Arandas.

How you feel about it: You really do not want to go there.

Write the one chat message you would send the group in reply. Say it plainly and directly.
Keep it to one or two short sentences, in your own voice. Do not mention
numbers, scores or ratings, and do not use the words "accept", "reject" or
"strong dislike". Reply with the message only.

feelings: {
 "accept": "You are happy to go there.",
 "reject": "You would mildly prefer somewhere else, but it is not a big deal.",
 "strong_dislike": "You really do not want to go there.",
 "veto": "You refuse to go there at all, and you are using your one veto to block it. Say that you are using your veto."
}
styles: {
 "plain": "Say it

In [16]:
LIVE_ARMS = [("  + live AI diner", dict(react_fn=rev_graded)),
             ("  + live AI diner + veto token", dict(react_fn=rev_graded, veto_remove=True, veto_grad=True))]


def truths(city, kind):
    # E5's draws, in E5's order, for all 200 groups (a pilot then takes the first few)
    rng0 = np.random.default_rng(11)
    Us = [truth(g, kind, rng0) for g in D[city]]
    return Us, [U.min(1).max() - REV_MARGIN for U in Us]


def live_run(city, kind, style, arm, kw, n=None):
    # one arm over the first n groups; returns the loop results and each group's messages
    G = D[city][:n]
    Us, bars = truths(city, kind)
    names = catalog_names(city)

    def one(j):
        g, recs = G[j], []
        talk = CHAT.talker(city=city, truth=kind, arm=arm, style=style, group_id=g["group_id"],
                           members=list(range(g["n"])), option_name=[names[o] for o in g["ids"]],
                           records=recs)
        return rev_mediate(g["S"], U=Us[j], bar=bars[j], talk=talk, **kw), recs

    with ThreadPoolExecutor(LIVE_WORKERS) as ex:
        out = list(ex.map(one, range(len(G))))
    return [r for r, _ in out], [m for _, m in out], Us[:len(G)], bars[:len(G)]


def spend():
    calls = ", ".join(f"{k} {v}" for k, v in CHAT.calls.items())
    toks = ", ".join(f"{m} {a:,} in / {b:,} out" for m, (a, b) in CHAT.usage.items())
    print(f"new calls this session: {calls}" + (f" | tokens: {toks}" if toks else ""))
    if MEDIATOR_LIVE:
        print(nbio.openai_spend())

### Pilot: the first 5 Houston groups, exact reactions

Every writing style, with and without the veto token. Read the conversations
before trusting any number below: the plain-style chats are printed in full, the
other styles for the first two groups. Writes `figdata_rev/e8_live_llm_pilot.csv`.

In [17]:
pilot_rows, PILOT = [], {}
try:
    for style in PILOT_STYLES:
        for arm, kw in LIVE_ARMS:
            res, msgs, Us, bars = live_run("Houston, TX", "proxy is exact", style, arm, kw, n=PILOT_N)
            PILOT[(style, arm)] = (res, msgs)
            flat = [m for ms in msgs for m in ms]
            pilot_rows.append(rev_summarise(res, Us, bars, arm, "Houston, TX",
                                            dict(style=style, **rev_talk_stats(flat))))
except LookupError as e:
    print(f"pilot not complete in the log ({e}); set MEDIATOR_LIVE = True to run it")
if pilot_rows:
    write("e8_live_llm_pilot.csv", pilot_rows)
nbio.table(pilot_rows, cols=["style", "arm", "pct_agree", "rounds_capped", "n_messages",
                             "pct_reader_wrong", "pct_wrong_objections", "pct_wrong_accepts",
                             "n_member_wrong"], width=30)
spend()

style      arm                             pct_agree  rounds_capped  n_messages  pct_reader_wrong  pct_wrong_objections  pct_wrong_accepts  n_member_wrong
----------------------------------------------------------------------------------------------------------------------------------------------------------
plain        + live AI diner               60.000     5.600          115         21.700            100.000               0.000              0             
plain        + live AI diner + veto token  60.000     4.400          97          14.400            73.700                0.000              0             
polite       + live AI diner               60.000     5.600          115         21.700            100.000               0.000              0             
polite       + live AI diner + veto token  60.000     5.000          106         17.900            86.400                0.000              0             
vague        + live AI diner               60.000     5.600          1

In [18]:
for (style, arm), (res, msgs) in PILOT.items():
    show = range(len(msgs)) if style == "plain" else range(min(2, len(msgs)))
    for j in show:
        g = D["Houston, TX"][j]
        print(f"=== {style} | {arm.strip()} | {g['group_id']} ({g['n']} diners): "
              f"{res[j]['reason']} after {res[j]['rounds']} round(s)")
        print_chat(msgs[j])
        print()

=== plain | + live AI diner | g0000 (5 diners): cap after 8 round(s)
  round 1: mediator suggests The Teahouse Tapioca & Tea (Chinatown)
    Ann   meant strong_dislike read reject           <-- MISREAD
          | I’m not really feeling The Teahouse Tapioca & Tea. Can we
          | consider another option?
    Ben   meant accept         read accept        
          | Sounds great to me! I'm happy to go to The Teahouse Tapioca
          | & Tea.
    Cara  meant accept         read accept        
          | The Teahouse Tapioca & Tea sounds great to me! I'm happy to
          | go there.
    Dev   meant accept         read accept        
          | Sounds great! I'm happy to go to The Teahouse Tapioca & Tea.
    Eli   meant accept         read accept        
          | Sounds great to me! I'm happy to go to The Teahouse Tapioca
          | & Tea.
  round 2: mediator suggests The Teahouse Tapioca & Tea (Chinatown)
    Ann   meant strong_dislike read reject           <-- MISREAD
     

### The full run: 200 groups, Houston and Seattle, every truth and style

Writes `figdata_rev/e8_live_llm.csv`: the usual mediation columns (agreement %,
rounds, final MinSat / mean / Jain), plus `truth` and `style`, and the reader's
error rate. `pct_reader_wrong` is over all messages; `pct_wrong_objections` and
`pct_wrong_accepts` split it by what the diner meant. `n_objection_read_as_accept`
counts complaints the loop never heard. `n_fallback` counts readings that fell
back to `reject` at confidence 0, and `n_member_wrong` counts readings that
attributed the message to someone else.

In [19]:
rows8, missing = [], []
FULL_GRID = REV_CITIES if (RUN_FULL or not MEDIATOR_LIVE) else []   # live, pilot only: skip
if not FULL_GRID:
    print("RUN_FULL is False: the full grid is not generated in this live session")
for city in FULL_GRID:
    for kind in LIVE_TRUTHS:
        for style in LIVE_STYLES:
            for arm, kw in LIVE_ARMS:
                try:
                    res, msgs, Us, bars = live_run(city, kind, style, arm, kw)
                except LookupError:
                    missing.append((city, kind, style, arm.strip()))
                    continue
                flat = [m for ms in msgs for m in ms]
                rows8.append(rev_summarise(res, Us, bars, arm, city,
                                           dict(truth=kind, style=style, **rev_talk_stats(flat))))
                print({k: rows8[-1][k] for k in ("city", "truth", "style", "arm", "pct_agree",
                                                 "rounds_capped", "pct_reader_wrong")})
if missing:
    print(f"\n{len(missing)} run(s) not in the log yet, e.g. {missing[0]}; set MEDIATOR_LIVE = True to run them")
if rows8:
    write("e8_live_llm.csv", rows8)
spend()


64 run(s) not in the log yet, e.g. ('Houston, TX', 'proxy is exact', 'plain', '+ live AI diner'); set MEDIATOR_LIVE = True to run them
new calls this session: writer 0, reader 0


In [20]:
# side by side with E5 (Table 8): agreement % per truth, Houston / Seattle. The
# replay rows are E5's from this session (rows5), so both come from the same matrices.
if rows8:
    e5 = pd.DataFrame(rows5)
    e8 = pd.DataFrame(rows8)
    replay = {"  + live AI diner": "  + LLM parser replay",
              "  + live AI diner + veto token": "  + LLM parser replay + veto token"}
    show = []
    for arm in replay:
        for src, lab, df_ in [("replay", replay[arm], e5)] + [(s, arm, e8[e8["style"] == s]) for s in LIVE_STYLES]:
            r = {"arm": arm.strip(), "reader": src}
            for t in LIVE_TRUTHS:
                cell = []
                for c in REV_CITIES:
                    x = df_[(df_.city == c) & (df_.arm == lab) & (df_.truth == t)]
                    cell.append(f"{x.pct_agree.iloc[0]:.1f}" if len(x) else "--")
                r[t] = " / ".join(cell)
            show.append(r)
    nbio.table(show, width=26)